# SOLUTIONS — Assignment: LangChain Fundamentals (Landscape through Tools)

**Instructor/answer-key copy.** Matches `Assignment_LangChain_Fundamentals_Landscape_to_Tools.ipynb`
question for question. Part A answers are written out in full below each question. Part B and
Part C provide one complete, verified working solution each — genuinely correct, not the only
possible correct answer, since several exercises (especially the capstone) are intentionally
open-ended.

**Keep this file separate from what you hand students** — the whole point of the assignment is
that they work through it themselves first.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY -- check your .env file or Colab Secrets"

from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-5-mini")
print("Environment ready.")


---
# Part A — Conceptual Answers

## A1. Foundations

**1.** "Model" is the raw language model — extraordinary reasoning ability, but no way to act on
the world by itself. "Harness" is everything that turns that raw model into something useful:
the system prompt (instructions), the tools it can reach for, and any middleware shaping its
behavior at each step. `create_agent` *is* the harness — configuring it is what almost this
entire course teaches.

**2.** LangGraph = low-level orchestration, the foundation everything else sits on. LangChain
(`create_agent`) = a configurable harness built on LangGraph — what this course is built around.
Deep Agents = a batteries-included harness on top of `create_agent` (planning, filesystem,
subagents pre-wired). LangSmith = fundamentally different in kind — it's an *observability*
platform, not something you build agents *with*; it watches agents built with any of the other
three, via traces, since an agent's runtime decisions can't be read from its code alone.

**3.** Conclude the content predates October 2025 (LangChain v1.0) — `AgentExecutor` and
`initialize_agent` are the legacy pre-v1.0 pattern, now living in a separate `langchain-classic`
package. Use `create_agent` instead.

**4.** `.env` keeps a real API key out of the notebook file itself. Hardcoding it directly means
the moment that notebook is committed to GitHub or shared, the key is exposed — often found and
abused by automated scanners within minutes.

**5.** Plain text: a single, standalone request, no history needed. Message objects: multi-turn
conversations, multimodal content, explicit system instructions. Dictionaries: functionally the
same as message objects, natural when conversation data arrives from a database or JSON API.

## A2. Models, Messages, and Templates

**6.** `.tool_calls` (any tool requests), `.usage_metadata` (real token counts), `.id` (unique
identifier), `.response_metadata` (provider-specific extras like which exact model responded).
`.content_blocks` is also acceptable (standardized rich content structure).

**7.** Streaming needs to represent a PARTIAL response that can later be combined into a whole —
`AIMessageChunk` objects support being summed with `+` into a complete `AIMessage`, carrying
structure (potential tool call fragments, metadata) that a plain string fragment can't.

**8.** `.batch()` waits for ALL results and returns them in input order. `.batch_as_completed()`
yields each result as soon as IT individually finishes, possibly out of order (each tagged with
its original input index). Use the second when you want to start using results as soon as any
one finishes, without waiting on the slowest one — e.g., processing a batch of unrelated
independent requests where whichever comes back first should be handled first.

**9.** `.content` is what the MODEL reads. `.artifact` is extra data the APPLICATION can use,
never sent to the model. A RAG tool wants this to attach a document ID or citation link the UI
needs, without bloating what the model has to process.

**10.** LangChain tries to interpret the literal `{` and `}` in the JSON example as MORE template
variables, and fails (`INVALID_PROMPT_INPUT`). Fix: escape by doubling the braces — `{{` and `}}`.

**11.** `MessagesPlaceholder` injects an entire LIST of prior messages as one template variable.
A string variable can only hold a single string value — it has no way to represent a whole
structured message history.

## A3. Structured Output

**12.** Plain prompting relies on the model CHOOSING to follow an instruction — it can still
wander, add commentary, or produce invalid JSON. `with_structured_output()` gives the model an
actual schema validated by Pydantic before the result ever reaches your code; non-compliant
output doesn't just pass through as "close enough."

**13.** `model.profile` is a real dict describing what a specific model actually supports (e.g.
`"structured_output": true`, `"tool_calling": true`). When no strategy is specified explicitly,
LangChain checks this profile to auto-select `ProviderStrategy` (if supported) or fall back to
`ToolStrategy`.

**14.** `strategy=` is not a real parameter on `with_structured_output()` — it gets silently
absorbed into `**kwargs` and does nothing, so the call quietly behaves as the default
auto-selected strategy instead of what was intended. The schema itself should be wrapped:
`model.with_structured_output(ProviderStrategy(BookingRequest))`.

**15.** Model-level (`with_structured_output`) has no awareness of tools or a tool-calling loop —
it just returns a parsed object directly. Agent-level (`response_format` on `create_agent`)
coexists with tools, returning `result["structured_response"]` alongside the normal message
trace. Almost everything from Part 7 onward uses `create_agent`, so the agent-level version is
what's actually needed in practice.

**16.** The model itself decides which of the Union's member schemas actually fits the message,
based on its own judgment of the content — LangChain doesn't pre-classify it. If the message is
genuinely ambiguous, the model may even attempt to satisfy more than one schema before settling,
which connects directly to the self-correction behavior in Q17.

**17.** The model first proposes `party_size=50`. Pydantic validates it against `ge=1, le=20` and
rejects it. That validation error — naming the exact constraint violated — is fed back to the
model as a message. The standard agent loop (unmodified) lets the model see that error and
retry with a corrected value, repeating until a value satisfies the constraint or `handle_errors`
gives up according to its configured behavior.

## A4. Tools

**18.** Largely defensible — the docstring genuinely is what the model reads to decide whether
and when to call a tool. A case where it matters less: a tool that's the ONLY option available
and always needed on every turn — but even then, a bad docstring risks the model second-guessing
whether to call it at all, so it's rarely truly irrelevant.

**19.** `ToolRuntime` hides `state`, `context`, `store`, `execution_info`, and `server_info` from
the model — none of these appear in the tool's schema. LangChain detects this by recognizing the
`ToolRuntime` TYPE ANNOTATION on the parameter named `runtime`, and automatically excludes it
when building what the model sees.

**20.** `runtime.state`: this conversation only, mutable, gone once the thread ends (e.g. what
the customer already said). `runtime.context`: immutable, set once per invocation, must be
passed fresh every call (e.g. a user's membership tier for this specific request).
`runtime.store`: survives across ANY number of separate sessions (e.g. a saved dietary
preference, recalled weeks later).

**21.** `config` is reserved internally by the framework for `RunnableConfig`. Using it as a
regular argument name causes the framework to intercept it before the function body ever runs,
typically producing a `TypeError` about a missing required argument. Easy to hit by accident
because `config` is an unremarkable, common variable name with no obvious reason to expect it's
reserved.

**22.** A plain string only ever ANSWERS a question. `Command` can write directly to agent STATE.
Original example: a tool that upgrades a customer's loyalty tier after a large order — it needs
to both confirm the upgrade AND persist the new tier into state for later tools/turns to read;
a plain string return has no mechanism to change anything beyond the current reply.

**23.** A system-prompt instruction relies on the model CHOOSING to comply — a sufficiently
persuasive or unusual phrasing could still talk it into trying. `wrap_model_call`-based gating
removes the tool from `request.tools` entirely before the model call happens — there's no tool
definition left for the model to reference at all, making it a structural guarantee, not a
request.

**24.** A headless tool has its schema registered on the server (so the model can call it
normally), but its actual EXECUTION happens on the CLIENT — typically a browser — via an
interrupt/resume handshake, rather than running in the Python process. Example: reading the
user's real-time geolocation from the browser's Geolocation API — a Python server has no way to
access that directly; only client-side code can.


---
# Part B — Solutions

## B1.1


In [ ]:
from langchain_core.tools import tool

@tool
def check_table_availability(party_size: int, time_slot: str) -> str:
    """Check whether a table is available for a given party size and time slot."""
    available_slots = {2: ["7:00 PM", "8:30 PM"], 4: ["7:00 PM", "8:30 PM"], 6: ["7:00 PM", "8:30 PM"]}
    slots_for_size = available_slots.get(party_size, [])
    if time_slot in slots_for_size:
        return f"Table for {party_size} is available at {time_slot}."
    return f"No table for {party_size} available at {time_slot}."

print("Name:       ", check_table_availability.name)
print("Description:", check_table_availability.description)
print("Args:       ", check_table_availability.args)


## B1.2

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

dish_prompt = ChatPromptTemplate.from_messages([
    ("system", "You generate a short, enthusiastic one-sentence description of a dish."),
    ("human", "Dish: {dish_name}, Cuisine: {cuisine_type}"),
])
chain = dish_prompt | model

for dish, cuisine in [("Butter Chicken", "Indian"), ("Margherita Pizza", "Italian")]:
    result = chain.invoke({"dish_name": dish, "cuisine_type": cuisine})
    print(f"[{dish}] -> {result.content}\n")


## B1.3

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class FoodOrder(BaseModel):
    customer_name: str
    dish_name: str
    quantity: int = Field(ge=1, le=10)
    spice_level: Literal["mild", "medium", "hot"]

structured_model = model.with_structured_output(FoodOrder)
result = structured_model.invoke("Hi, I'm Karan, 2 butter chicken please, medium spice.")
print(result)


## B2.1

In [ ]:
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class NewReservation(BaseModel):
    customer_name: str
    party_size: int
    time_slot: str

class CancelReservation(BaseModel):
    customer_name: str
    time_slot: str

reservation_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[],
    response_format=ToolStrategy(Union[NewReservation, CancelReservation]),
)

r1 = reservation_agent.invoke({"messages": [{"role": "user", "content": "Table for 4 at 7pm, name is Aisha"}]})
print("Message 1 ->", type(r1["structured_response"]).__name__, "|", r1["structured_response"])

r2 = reservation_agent.invoke({"messages": [{"role": "user", "content": "Cancel my 7pm reservation, I'm Aisha"}]})
print("Message 2 ->", type(r2["structured_response"]).__name__, "|", r2["structured_response"])

print()
print("isinstance checks:")
print(" r1 is NewReservation:", isinstance(r1["structured_response"], NewReservation))
print(" r2 is CancelReservation:", isinstance(r2["structured_response"], CancelReservation))


## B2.2

In [ ]:
class OrderInput(BaseModel):
    dish_name: str = Field(description="The name of the dish being ordered")
    quantity: int = Field(description="Number of units to order", ge=1, le=10)
    delivery_or_pickup: Literal["delivery", "pickup"] = Field(default="pickup", description="Fulfillment method")

@tool(args_schema=OrderInput)
def place_order(dish_name: str, quantity: int, delivery_or_pickup: str = "pickup") -> str:
    """Place a food order."""
    return f"Order placed: {quantity}x {dish_name}, {delivery_or_pickup}."

print(place_order.args)


## B2.3

In [ ]:
strict_order_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[],
    response_format=ToolStrategy(FoodOrder),
    system_prompt="Extract the order exactly as stated. Do not invent information.",
)

# "15 units" deliberately violates FoodOrder's quantity constraint (ge=1, le=10)
result = strict_order_agent.invoke({"messages": [{"role": "user", "content": "I'm Rohan, order 15 samosas, mild spice."}]})

for m in result["messages"]:
    print(f"--- {m.type} ---")
    print(m.content if isinstance(m.content, str) else m.tool_calls)
# The self-correction happens where a ToolMessage/error naming the ge=1,le=10 violation
# appears in the trace above, immediately followed by the model retrying with a valid quantity.

print()
print("Final structured_response:", result["structured_response"])


## B3.1

In [ ]:
from typing import Any
from langchain.tools import tool as tool_rt, ToolRuntime
from langgraph.store.memory import InMemoryStore

pref_store = InMemoryStore()

@tool_rt
def save_dietary_preference(customer_id: str, preference: str, runtime: ToolRuntime) -> str:
    """Save a customer's dietary preference for future visits."""
    runtime.store.put((customer_id, "preferences"), "dietary", {"value": preference})
    return f"Saved: {preference}."

@tool_rt
def recall_dietary_preference(customer_id: str, runtime: ToolRuntime) -> str:
    """Recall a customer's saved dietary preference."""
    result = runtime.store.get((customer_id, "preferences"), "dietary")
    return result.value["value"] if result else "No preference saved yet."

memory_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[save_dietary_preference, recall_dietary_preference],
    store=pref_store,
)

# Call 1: save
memory_agent.invoke({"messages": [("user", "I'm customer aisha_01, I'm vegetarian, please remember that.")]})

# Call 2: a SEPARATE .invoke() call -- proves this is genuine cross-session memory
result = memory_agent.invoke({"messages": [("user", "What's my dietary preference? I'm aisha_01.")]})
print(result["messages"][-1].content)


## B3.2

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@tool
def book_private_dining_room(party_size: int) -> str:
    """Book the private dining room. Premium members only."""
    return f"Private dining room booked for {party_size} guests."

@wrap_model_call
def gate_private_dining(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only expose book_private_dining_room to premium members."""
    is_premium = request.state.get("is_premium_member", False)
    if not is_premium:
        allowed = [t for t in request.tools if t.name != "book_private_dining_room"]
        request = request.override(tools=allowed)
    return handler(request)

gated_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[check_table_availability, book_private_dining_room],
    middleware=[gate_private_dining],
)

r_regular = gated_agent.invoke({"messages": [("user", "Book the private dining room for 8 people")]})
print("Non-premium:", r_regular["messages"][-1].content)

r_premium = gated_agent.invoke(
    {"messages": [("user", "Book the private dining room for 8 people")], "is_premium_member": True}
)
print("Premium:    ", r_premium["messages"][-1].content)


## B3.3

In [ ]:
class ReservationConfirmation(BaseModel):
    customer_name: str
    time_slot: str
    confirmed: bool

combined_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[check_table_availability],
    response_format=ReservationConfirmation,
    system_prompt="Check availability with the tool before confirming a reservation.",
)

result = combined_agent.invoke({
    "messages": [{"role": "user", "content": "Table for 4 at 7:00 PM, name is Priya. Please confirm."}]
})
print("Messages trace (last):", result["messages"][-1].content)
print("Structured response:  ", result["structured_response"])

# This required create_agent(response_format=...) rather than raw with_structured_output()
# because with_structured_output() has no tool-calling loop of its own -- it can't call
# check_table_availability at all before returning. Only the agent-level version coexists
# with a real tool-calling loop, letting the agent check availability BEFORE answering.


---
# Part C — Capstone Solution

**Design explanation:** this solution uses all SIX listed concepts (the assignment only
required five, but a solution key is a good place to show the complete picture). A structured
`ReservationRequest` schema (with a `Literal`-constrained field) captures what the customer
wants; `check_table_availability` and `book_table` are the two required tools; a dietary
preference is remembered long-term via `ToolRuntime.store`; the customer's name persists
short-term via a checkpointer and `thread_id`; `book_private_dining_room` is gated to premium
members only; and a `context_schema` carries the restaurant's location, read directly by a tool.


In [ ]:
from dataclasses import dataclass
from langgraph.checkpoint.memory import InMemorySaver

class ReservationRequest(BaseModel):
    customer_name: str
    party_size: int = Field(ge=1, le=20)
    time_slot: Literal["6:00 PM", "7:00 PM", "8:30 PM"]

@dataclass
class RestaurantContext:
    location: str

@tool
def book_table(customer_name: str, party_size: int, time_slot: str) -> str:
    """Book a table for a customer."""
    return f"Table booked for {customer_name}, party of {party_size}, at {time_slot}."

@tool_rt
def get_location_info(runtime: ToolRuntime) -> str:
    """Get the restaurant's current location, from per-run context."""
    return f"This GreenPlate location is in {runtime.context.location}."

capstone_store = InMemoryStore()

@wrap_model_call
def gate_private_dining_capstone(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    is_premium = request.state.get("is_premium_member", False)
    if not is_premium:
        allowed = [t for t in request.tools if t.name != "book_private_dining_room"]
        request = request.override(tools=allowed)
    return handler(request)

capstone_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[
        check_table_availability, book_table, book_private_dining_room,
        save_dietary_preference, recall_dietary_preference, get_location_info,
    ],
    system_prompt="You are GreenPlate's booking assistant.",
    middleware=[gate_private_dining_capstone],
    checkpointer=InMemorySaver(),
    store=capstone_store,
    context_schema=RestaurantContext,
    response_format=ReservationRequest,
)

config = {"configurable": {"thread_id": "capstone-session"}}

# Call 1: short-term memory -- introduce the customer's name
capstone_agent.invoke(
    {"messages": [("user", "My name is Meera, I'm vegan, please remember that.")]},
    config=config, context=RestaurantContext(location="Bandra, Mumbai"),
)

# Call 2: recall the name (short-term) AND the dietary preference (long-term, different system)
result = capstone_agent.invoke(
    {"messages": [("user", "What's my name, and what do I usually eat? Also, where is this location?")]},
    config=config, context=RestaurantContext(location="Bandra, Mumbai"),
)
print(result["messages"][-1].content)


**Reflection:** the biggest limitation here is that `InMemoryStore` and `InMemorySaver` both
lose everything the moment this process restarts — genuinely fine for a demo, not acceptable for
production. Before this went live, both would need to be swapped for their database-backed
equivalents (`PostgresStore`, `PostgresSaver`), which — as covered in the Memory module — is a
one-line change in which object gets constructed, not a rewrite of anything else here. The
`book_table` tool is also currently unguarded by `HumanInTheLoopMiddleware`, which a real
production booking system would likely want, given it's a genuinely consequential action.
